<a href="https://colab.research.google.com/github/oliveira-o2/AMORA/blob/main/AMORA_BASE_CONSOLIDADA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
def consolidar_arquivos():
    print("Buscando arquivos...")
    nome_saida = "Relatório de Itens por NFe - Consolidado.xlsx"

    # 1. Buscamos os arquivos, mas IGNORAMOS o arquivo consolidado se ele já existir
    arquivos = [f for f in (glob.glob("Relatório de Itens por NFe*.xls") +
                            glob.glob("Relatório de Itens por NFe*.xlsx"))
                if f != nome_saida]

    if not arquivos:
        print("Arquivos não encontrados na pasta atual.")
        return

    lista_df = []

    for f in arquivos:
        try:
            df = ler_excel(f)
            # ... (seu código de limpeza continua igual aqui)
            df = df[~df.iloc[:, 0].astype(str).str.contains("Total|Soma|Somatório", case=False, na=False)]
            colunas_c_em_diante = df.columns[2:]
            df = df.dropna(subset=colunas_c_em_diante, how='all')

            lista_df.append(df)
            print(f"✔ Arquivo processado: {f}")
        except Exception as e:
            print(f"❌ Erro ao processar o arquivo {f}: {e}")

    if not lista_df:
        print("Nenhum dado pôde ser extraído após a limpeza.")
        return

    print("\nConsolidando os dados...")
    df_final = pd.concat(lista_df, ignore_index=True)

    # ... (seu código de conversão de tipos continua igual aqui)
    df_final.iloc[:, 0] = pd.to_datetime(df_final.iloc[:, 0], errors='coerce', dayfirst=True)
    colunas_para_numero = [4] + list(range(12, 23))
    for col_idx in colunas_para_numero:
        if col_idx < len(df_final.columns):
            df_final.iloc[:, col_idx] = pd.to_numeric(df_final.iloc[:, col_idx], errors='coerce')

    # 2. Tratamento de erro específico para o arquivo aberto
    try:
        df_final.to_excel(nome_saida, index=False)
        print("Aplicando formatação no Excel (Fonte Arial, Datas e Números)...")
        formatar_excel(nome_saida)
        print(f"\n✅ Sucesso! Arquivo gerado: '{nome_saida}'")
    except PermissionError:
        print(f"\n❌ ERRO: O arquivo '{nome_saida}' está aberto no Excel.")
        print("Por favor, feche o arquivo e execute o script novamente.")